#### Data Ingestion Configuration

In [1]:
import os

In [2]:
%pwd

'c:\\Users\\DeLL\\OneDrive\\Desktop\\MLOPS Projects\\DataScience-end-to-end\\research'

In [3]:
os.chdir("../")
%pwd

'c:\\Users\\DeLL\\OneDrive\\Desktop\\MLOPS Projects\\DataScience-end-to-end'

In [10]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


In [11]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories

In [12]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.dataIngestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )
        return data_ingestion_config
    
    

In [13]:
import os
from urllib.request import urlretrieve
from src.datascience import logger
import zipfile

In [16]:
# Component, DataIngestion
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with the follwing info:\n {headers}")
        else:
            logger.info(f"{filename} already exists.")
    
    def extract_zip(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)

        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [17]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip()
except Exception as e:
    raise e

[2026-04-01 18:17:58,233: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-01 18:17:58,236: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-01 18:17:58,241: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-01 18:17:58,244: INFO: common: Created directory at: artifacts]
[2026-04-01 18:17:58,246: INFO: common: Created directory at: artifacts/data_ingestion]
[2026-04-01 18:17:59,192: INFO: 3987066220: artifacts/data_ingestion/data.zip downloaded with the follwing info:
 Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 0DF2:A6E67:393BB4:5AE050:69CD13FC
Accept-Ranges: bytes
Date: Wed,